# ISOM 835 · Session 4 — Regression: Predicting Numbers
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Oct 5 · Prof. Hasan Arslan**

What is a house worth? Linear regression as the interpretable baseline, coefficients in dollars, a log target, ridge / lasso / elastic net, and the residual plot that tells you what your model is missing.

> **Frame the prediction (Ames).** *Unit:* one house · *Target:* sale price · *Horizon:* at listing · *Decision:* list price / offer price · *Baseline:* the mean price.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error

ames = fetch_openml('house_prices', as_frame=True)
X, y = ames.data.drop(columns=['Id']), ames.target
print(X.shape, f'median ${y.median():,.0f}  mean ${y.mean():,.0f}')

## 1. The target, and the baseline
Sale price is right-skewed: a handful of $500K+ houses will dominate squared error. The baseline every model must beat is **predict the mean for everyone**.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=835)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
y_tr.plot(kind='hist', bins=40, ax=ax[0], color='#7c8cff', title='Sale price'); np.log1p(y_tr).plot(kind='hist', bins=40, ax=ax[1], color='#2ee6c5', title='log(1 + price)'); plt.show()
base_pred = np.full(len(y_te), y_tr.mean())
print(f'Baseline (mean): RMSE ${root_mean_squared_error(y_te, base_pred):,.0f}   MAE ${mean_absolute_error(y_te, base_pred):,.0f}')

## 2. One feature, one line
Price against above-ground living area. The slope is a number a realtor can use.

In [ ]:
lr1 = LinearRegression().fit(X_tr[['GrLivArea']], y_tr)
print(f'price ≈ {lr1.intercept_:,.0f} + {lr1.coef_[0]:,.1f} × sqft      test R² {r2_score(y_te, lr1.predict(X_te[["GrLivArea"]])):.3f}')
plt.figure(figsize=(6, 3.6)); plt.scatter(X_tr['GrLivArea'], y_tr, s=8, alpha=0.5, color='#7c8cff')
xs = np.linspace(300, 4000, 50); plt.plot(xs, lr1.predict(pd.DataFrame({'GrLivArea': xs})), color='#2ee6c5', lw=2); plt.xlabel('living area (sq ft)'); plt.ylabel('price'); plt.show()

## 3. Many features, log target — the Session 3 pipeline
Same `ColumnTransformer` recipe. Target = `log1p(price)`, so errors become *percentage* errors and the skew is tamed. Predict, then `expm1` back to dollars before reporting.

In [ ]:
num = X.select_dtypes(include='number').columns.tolist()
cat = [c for c in X.columns if c not in num]
prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')), ('ohe', OneHotEncoder(handle_unknown='ignore', min_frequency=10))]), cat),
])
def report(name, model, log_target=True):
    model.fit(X_tr, np.log1p(y_tr) if log_target else y_tr)
    p = model.predict(X_te); p = np.expm1(p) if log_target else p
    print(f'{name:22s} RMSE ${root_mean_squared_error(y_te, p):>8,.0f}   MAE ${mean_absolute_error(y_te, p):>7,.0f}   MAPE {mean_absolute_percentage_error(y_te, p):.1%}   R² {r2_score(y_te, p):.3f}')
    return p
ols_raw = report('OLS, raw target', Pipeline([('prep', prep), ('m', LinearRegression())]), log_target=False)
ols_log = report('OLS, log target', Pipeline([('prep', prep), ('m', LinearRegression())]))

Plain OLS on 200+ one-hot columns with 1,168 training rows is already flirting with overfitting — and the *raw-target* version is hurt badly by the expensive houses. Now the cure.

## 4. Watching overfitting, then regularizing
Polynomial features on a few numeric columns make train R² soar and test R² collapse — the Overfitting Explorer with real houses. Ridge, lasso, and elastic net shrink the coefficients; `…CV` variants choose alpha by cross-validation.

In [ ]:
few = ['GrLivArea', 'OverallQual', 'YearBuilt', 'TotalBsmtSF', 'GarageCars']
for deg in [1, 2, 3]:
    poly = make_pipeline(SimpleImputer(strategy='median'), PolynomialFeatures(deg, include_bias=False), StandardScaler(), LinearRegression())
    poly.fit(X_tr[few], np.log1p(y_tr))
    print(f'degree {deg}: train R² {r2_score(np.log1p(y_tr), poly.predict(X_tr[few])):.3f}   test R² {r2_score(np.log1p(y_te), poly.predict(X_te[few])):.3f}')

In [ ]:
alphas = np.logspace(-4, 1, 40)
ridge = report('RidgeCV', Pipeline([('prep', prep), ('m', RidgeCV(alphas=alphas))]))
lasso_pipe = Pipeline([('prep', prep), ('m', LassoCV(cv=5, random_state=835, max_iter=20000))])
lasso = report('LassoCV', lasso_pipe)
enet = report('ElasticNetCV', Pipeline([('prep', prep), ('m', ElasticNetCV(l1_ratio=[.2, .5, .8], cv=5, random_state=835, max_iter=20000))]))
coef = lasso_pipe.named_steps['m'].coef_
print(f'\nlasso alpha {lasso_pipe.named_steps["m"].alpha_:.4f} → {(coef != 0).sum()} of {len(coef)} coefficients survive')

## 5. Read the survivors
Lasso's non-zero coefficients live on the log scale: a coefficient of 0.05 on a standardized feature means roughly +5% price per standard deviation. The list should read like a realtor's checklist.

In [ ]:
names = lasso_pipe.named_steps['prep'].get_feature_names_out()
surv = pd.Series(coef, index=names); surv = surv[surv != 0].sort_values()
print('most negative:'); print(surv.head(6).round(3)); print('\nmost positive:'); print(surv.tail(8).round(3))

## 6. Residual diagnostics
Residuals vs. fitted: a **fan** means the errors grow with the prediction (heteroscedasticity — take the log), a **curve** means a missing nonlinearity, a **cluster of big misses** means a segment the model doesn't understand.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for a, (name, p) in zip(ax, [('OLS raw target', ols_raw), ('Lasso, log target', lasso)]):
    a.scatter(p, y_te - p, s=10, alpha=0.6, color='#ff6b8b'); a.axhline(0, color='#2ee6c5'); a.set_title(name); a.set_xlabel('predicted price'); a.set_ylabel('residual (actual − predicted)')
plt.tight_layout(); plt.show()
worst = (y_te - lasso).abs().sort_values(ascending=False).head(5)
print('largest misses:'); print(pd.DataFrame({'actual': y_te.loc[worst.index], 'predicted': lasso[[list(y_te.index).index(i) for i in worst.index]].round(0), 'GrLivArea': X_te.loc[worst.index, 'GrLivArea'], 'Neighborhood': X_te.loc[worst.index, 'Neighborhood']}))

## 7. The metric menu — which number for whom
| Metric | Reads as | Report it to |
|---|---|---|
| **MAE** | the typical miss, in dollars | operations, sales |
| **RMSE** | the typical miss, but big misses count extra | risk, finance |
| **MAPE** | the typical miss as a % — breaks near zero | executives (with care) |
| **R²** | share of variance explained vs. the mean | analysts |

Preview of Session 8: the same pipeline with `HistGradientBoostingRegressor` at the end.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
gbm = report('HistGB (preview)', Pipeline([('prep', prep), ('m', HistGradientBoostingRegressor(learning_rate=0.05, max_iter=1500, early_stopping=True, random_state=835))]))

## 8. Your turn
1. **Neighborhood premium.** Fit lasso *without* the `Neighborhood` column. How much does test RMSE change? What does that say about location?
2. **Interaction.** Add `GrLivArea × OverallQual` as a feature (inside the pipeline with `FunctionTransformer`). Does lasso keep it?
3. **Your own metric memo.** Write three sentences: which metric you would report to a home seller, to a mortgage lender, and to your data-science lead — and why.

In [ ]:
# Your turn — work here

## What we learned tonight
- A linear coefficient is a **slope in business units, holding the others fixed** — standardize before comparing, never read as a cause.
- **Log the target** when it is skewed and residuals fan out; predict, then `expm1` back to dollars.
- **Ridge shrinks, lasso selects.** Let CV choose alpha; scale first.
- **Look at residuals** — the plot tells you what the model is missing before any metric does.

**Homework #2** (due Mon Oct 19 — no class Oct 12): the Ames pipeline with ridge/lasso/elastic net, metrics in dollars, residual plot, and a paragraph on the worst segment. Bonus: submit to the Kaggle House Prices competition.